In [1]:
import pandas as pd
import numpy as np
import joblib

In [2]:
net_model = joblib.load("../models/random_forest_network.pkl")

multi_model = joblib.load("../models/random_forest_multiclass.pkl")

cloud_model = joblib.load("../models/cloud_random_forest.pkl")

host_model = joblib.load("../models/host_isolation_forest_10M.pkl")

print("All models loaded successfully")

All models loaded successfully


In [3]:
def fusion_decision(network_pred, cloud_pred, host_pred):
    
    score = 0

    # Network
    if network_pred != "Benign":
        score += 60

    # Cloud
    if cloud_pred == "Suspicious":
        score += 20

    # Host
    if host_pred == -1:
        score += 20

    # Final Label
    if score >= 70:
        level = "HIGH"
    elif score >= 40:
        level = "MEDIUM"
    elif score >= 20:
        level = "LOW"
    else:
        level = "NORMAL"

    return score, level

In [4]:
score, level = fusion_decision(
    network_pred="DDoS",
    cloud_pred="Normal",
    host_pred=-1
)

print("Risk Score:", score)
print("Alert Level:", level)

Risk Score: 80
Alert Level: HIGH


In [5]:
tests = [
    ("Benign", "Normal", 1),
    ("Benign", "Suspicious", 1),
    ("DDoS", "Normal", 1),
    ("DDoS", "Normal", -1),
    ("Benign", "Suspicious", -1)
]

for t in tests:
    score, level = fusion_decision(*t)
    print(t, "=>", score, level)

('Benign', 'Normal', 1) => 0 NORMAL
('Benign', 'Suspicious', 1) => 20 LOW
('DDoS', 'Normal', 1) => 60 MEDIUM
('DDoS', 'Normal', -1) => 80 HIGH
('Benign', 'Suspicious', -1) => 40 MEDIUM


In [6]:
rows = []

for t in tests:
    score, level = fusion_decision(*t)

    rows.append({
        "Network": t[0],
        "Cloud": t[1],
        "Host": t[2],
        "Score": score,
        "Alert": level
    })

df = pd.DataFrame(rows)
df

,Network,Cloud,Host,Score,Alert
0,Benign,Normal,1,0,NORMAL
1,Benign,Suspicious,1,20,LOW
2,DDoS,Normal,1,60,MEDIUM
3,DDoS,Normal,-1,80,HIGH
4,Benign,Suspicious,-1,40,MEDIUM


In [8]:
import pandas as pd
import joblib

host_model = joblib.load("../models/host_isolation_forest_10M.pkl")

In [9]:
host_sample = pd.read_json(
    "../Data/host/wls_day-39",
    lines=True,
    nrows=50000
)

host_sample.head()

,UserName,EventID,LogHost,LogonID,DomainName,ParentProcessName,ParentProcessID,ProcessName,Time,ProcessID,...,AuthenticationPackage,LogonType,Source,Status,ServiceName,Destination,SubjectUserName,SubjectLogonID,SubjectDomainName,FailureReason
0,Comp785407$,4688,Comp785407,0x3e7,Domain001,services,0x2f0,rundll32.exe,3283200,0x1328,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,system,4624,Comp785407,0x3e7,nt authority,NaN,NaN,services.exe,3283200,0x2f0,...,Negotiate,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,system,4672,Comp785407,0x3e7,nt authority,NaN,NaN,NaN,3283200,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Comp604549$,4688,Comp604549,0x3e7,Domain001,svchost,0x47c,taskeng.exe,3283200,0x1390,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Comp686564$,4688,Comp686564,0x3e7,Domain001,svchost,0x3ac,dllhost.exe,3283200,0x770,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
X_host = pd.DataFrame()

X_host["EventID"] = host_sample["EventID"]
X_host["Hour"] = host_sample["Time"] % 24
X_host["Is_Failed_Logon"] = (host_sample["EventID"] == 4625).astype(int)
X_host["Is_Process_Create"] = (host_sample["EventID"] == 4688).astype(int)
X_host["Is_Privilege"] = (host_sample["EventID"] == 4672).astype(int)

In [11]:
host_pred = host_model.predict(X_host)

host_pool = pd.DataFrame({
    "host_pred": host_pred
})

print(host_pool["host_pred"].value_counts())

host_pred
 1    48963
-1     1037
Name: count, dtype: int64


In [12]:
cloud_model = joblib.load("../models/cloud_random_forest.pkl")

In [13]:
cloud = pd.read_csv("../Data/cloud/dataset_final_testset_merged.csv")
print(cloud.shape)

(36029, 10)


In [14]:
cloud["Hour"] = pd.to_datetime(cloud["Time"]).dt.hour
cloud["Message_Length"] = cloud["Message"].astype(str).apply(len)

X_cloud = cloud[["PID", "Module", "Label", "Hour", "Message_Length"]].copy()

X_cloud = pd.get_dummies(X_cloud, columns=["Module", "Label"])

C:\Users\Hamad Ali\AppData\Local\Temp\ipykernel_11856\589822490.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  cloud["Hour"] = pd.to_datetime(cloud["Time"]).dt.hour


In [16]:
train_cols = cloud_model.feature_names_in_

print("Expected columns:", len(train_cols))

Expected columns: 26


In [17]:
X_cloud = pd.get_dummies(X_cloud)

# Add missing columns
for col in train_cols:
    if col not in X_cloud.columns:
        X_cloud[col] = 0

# Remove extra unseen columns
X_cloud = X_cloud[train_cols]

In [18]:
cloud_pred = cloud_model.predict(X_cloud)

cloud_pool = pd.DataFrame({
    "cloud_pred": cloud_pred
})

print(cloud_pool["cloud_pred"].value_counts())

cloud_pred
Normal        18203
Suspicious    17826
Name: count, dtype: int64


In [19]:
net_model = joblib.load("../models/random_forest_network.pkl")
multi_model = joblib.load("../models/random_forest_multiclass.pkl")

In [20]:
net = pd.read_parquet("../Data/network/DDoS-Friday-no-metadata.parquet")

print(net.shape)

(221264, 78)


In [21]:
features = [
    'Protocol',
    'Flow Duration',
    'Total Fwd Packets',
    'Total Backward Packets',
    'Fwd Packets Length Total',
    'Bwd Packets Length Total',
    'Flow Bytes/s',
    'Flow Packets/s',
    'Packet Length Mean',
    'Packet Length Std',
    'SYN Flag Count',
    'ACK Flag Count'
]

X_net = net[features].copy()
X_net = X_net.fillna(0)

In [23]:
net_cols = net_model.feature_names_in_

print("Total expected features:", len(net_cols))
print(net_cols)

Total expected features: 16
['Protocol' 'Flow Duration' 'Total Fwd Packets' 'Total Backward Packets'
 'Fwd Packets Length Total' 'Bwd Packets Length Total' 'Flow Bytes/s'
 'Flow Packets/s' 'Packet Length Mean' 'Packet Length Std'
 'SYN Flag Count' 'ACK Flag Count' 'RST Flag Count' 'Avg Packet Size'
 'Active Mean' 'Idle Mean']


In [24]:
X_net = net.copy()

# Add missing columns if needed
for col in net_cols:
    if col not in X_net.columns:
        X_net[col] = 0

# Keep exact order
X_net = X_net[net_cols]

# Fill NaN / inf
X_net = X_net.replace([np.inf, -np.inf], 0)
X_net = X_net.fillna(0)

In [25]:
net_pred = net_model.predict(X_net)

pd.Series(net_pred).value_counts()

DDoS      127935
Benign     93329
Name: count, dtype: int64

In [26]:
network_pool = pd.DataFrame({
    "network_pred": net_pred
})

print(network_pool["network_pred"].value_counts())

network_pred
DDoS      127935
Benign     93329
Name: count, dtype: int64


In [27]:
net_vals = network_pool["network_pred"].values
cloud_vals = cloud_pool["cloud_pred"].values
host_vals = host_pool["host_pred"].values


In [28]:
import numpy as np
import pandas as pd

rows = []

for _ in range(1000):
    
    n = np.random.choice(net_vals)
    c = np.random.choice(cloud_vals)
    h = np.random.choice(host_vals)

    score, level = fusion_decision(n, c, h)

    rows.append({
        "Network": n,
        "Cloud": c,
        "Host": h,
        "Score": score,
        "Alert": level
    })

fusion_df = pd.DataFrame(rows)

fusion_df.head()

,Network,Cloud,Host,Score,Alert
0,DDoS,Normal,1,60,MEDIUM
1,DDoS,Normal,1,60,MEDIUM
2,DDoS,Normal,1,60,MEDIUM
3,Benign,Normal,1,0,NORMAL
4,Benign,Suspicious,1,20,LOW


In [29]:
print(fusion_df["Alert"].value_counts())

Alert
HIGH      309
MEDIUM    286
LOW       212
NORMAL    193
Name: count, dtype: int64


In [30]:
print("Average Risk Score:", fusion_df["Score"].mean())

Average Risk Score: 46.26


In [31]:
fusion_df[fusion_df["Alert"] == "HIGH"].head(20)

,Network,Cloud,Host,Score,Alert
6,DDoS,Suspicious,1,80,HIGH
7,DDoS,Suspicious,-1,100,HIGH
19,DDoS,Suspicious,1,80,HIGH
22,DDoS,Suspicious,1,80,HIGH
26,DDoS,Suspicious,1,80,HIGH
27,DDoS,Suspicious,1,80,HIGH
29,DDoS,Suspicious,1,80,HIGH
30,DDoS,Suspicious,1,80,HIGH
33,DDoS,Suspicious,1,80,HIGH
34,DDoS,Suspicious,-1,100,HIGH


In [1]:
import os

print(os.listdir("../models"))

['cloud_random_forest.pkl', 'host_isolation_forest.pkl', 'host_isolation_forest_10M.pkl', 'isolation_forest_network.pkl', 'network_scaler.pkl', 'random_forest_multiclass.pkl', 'random_forest_network.pkl']


In [3]:
import pandas as pd

sample = pd.read_json(
    "../Data/host/wls_day-39",
    lines=True,
    nrows=50000
)

sample.to_csv("../Data/host_sample_50k.csv", index=False)

print("Host sample saved")

Host sample saved


In [4]:
net = pd.read_parquet("../Data/network/DDoS-Friday-no-metadata.parquet")

net.head(50000).to_csv("../Data/network_sample_50k.csv", index=False)

print("Network sample saved")

Network sample saved


In [5]:
cloud = pd.read_csv("../Data/cloud/dataset_final_testset_merged.csv")

cloud.head(50000).to_csv("../Data/cloud_sample_50k.csv", index=False)

print("Cloud sample saved")

Cloud sample saved
